# 05 - Imitation as PPO Pretraining (Stage 5)

**Group members:** TODO

Central question: can imitation learning reduce the sample complexity of PPO? Compare five methods on return vs environment timesteps: PPO from scratch, BC only, DAgger only, BC + PPO fine-tune, DAgger + PPO fine-tune. Headline metric is timesteps to reach a target return.

In [ ]:
# Make src importable whether run from notebooks/ or project root
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
# On Colab: mount Drive and set PROJECT_DATA_ROOT before importing src
from src import config, seeding, envs, collect, eval, plotting
seeding.set_seed(0)
ENV_ID = 'Walker2d-v4'  # switch to 'Ant-v4' for the second environment
print('device', config.device(), '| env', ENV_ID)

In [ ]:
from src import bc_bridge
data = collect.load(config.DATA_DIR / ENV_ID)
DEVICE = config.device()
# 1) Train BC student (SB3 policy) via imitation
trainer, env = bc_bridge.train_bc_imitation(data['observations'], data['actions'],
                                            ENV_ID, n_epochs=50, device=DEVICE)
# 2) Warm-start PPO from the BC policy and fine-tune
vec_env = envs.make_vec(ENV_ID, n_envs=4, seed=0)
model = bc_bridge.ppo_from_policy(trainer.policy, vec_env, seed=0, device=DEVICE)
# TODO: model.learn(...) with eval logging; compare vs PPO-from-scratch curve